In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib
import numpy as np
import matplotlib.pyplot as plt
import joblib
import anndata as ad
from scipy.signal import find_peaks
import h5py



In [2]:
prot = joblib.load("C:\\Users\\olesk\\data\\all_prot_0318.sav")
umap_embed = joblib.load("C:\\Users\\olesk\\data\\embedding_0318.sav")
prot_d = joblib.load("C:\\Users\\olesk\\data\\all_prot_cycdi_0318.sav")
umap_d_embed = joblib.load("C:\\Users\\olesk\\data\\embedding_cycdi_0318.sav")

In [3]:
def short_ab(prot):
    prot_ab = prot[:,int((win_len*0+win_len/2)-2):int((win_len*0+win_len/2)+1)]
    for n in range(1,18):
        tmp_prot  = prot[:,int((win_len*n+win_len/2)-2):int((win_len*n+win_len/2)+1)]
        prot_ab = np.concatenate((prot_ab, tmp_prot), axis=1)
        
    return prot_ab

In [7]:
prot_ab = short_ab(prot)
prot_d_ab = short_ab(prot_d)

In [8]:
prot_ab = np.concatenate((prot_ab, prot_d_ab), axis=0)

In [6]:
len_prot = int(prot.shape[0]/6)
win_len = int(prot.shape[1]/18)

In [9]:
prot_ab = np.float32(prot_ab)

In [10]:
umap_embed = np.concatenate((umap_embed, umap_d_embed), axis=0)

In [11]:
adata = ad.AnnData(prot_ab)

In [12]:
adata.obs_names = [f"win_{i:d}" for i in range(adata.n_obs)]

In [13]:
prot_names = ["skp2", "p21", "pp21", "cdh1", "myc", "cycD", "cycD-p21", 
        "Rb", "pRb", "E2F", "E2F-Rb", "cycE", "cycE-p21", "cycA", "cycA-p21",
        "cycB", "cycB-p21", "cdc20"]

all_names = []
for name in prot_names:
    tmp_name = [f"{name}_{i:d}" for i in range(win_len)]
    all_names = all_names + tmp_name

In [14]:
name20 = []
for n in range(18):
    tmp_prot = [all_names[int((win_len*n+win_len/2)-2)], all_names[int((win_len*n+win_len/2)-1)], all_names[int((win_len*n+win_len/2))]]
    name20.extend(tmp_prot)

In [15]:
adata.var_names = name20

In [16]:
dt = ['DMSO']*len_prot + ['ERL']*len_prot + ['DMSO']*len_prot + ['ERL']*len_prot + ['DMSO']*len_prot + ['ERL']*len_prot + ['DMSO']*len_prot + ['ERL']*len_prot + ['DMSO']*len_prot + ['ERL']*len_prot + ['DMSO']*len_prot + ['ERL']*len_prot


In [17]:
adata.obs["drug"] = pd.Categorical(dt)

In [18]:
st = [0.2]*len_prot*2 + [4]*len_prot*2 + [55]*len_prot*2 + [0.2]*len_prot*2 + [4]*len_prot*2 + [55]*len_prot*2


In [19]:
adata.obs["stiffness"] = pd.Categorical(st)

In [20]:
inhib = ['none']*len_prot*6 + ['cycDi']*len_prot*6

In [21]:
adata.obs["inhibitor"] = pd.Categorical(inhib)

In [22]:
adata.obsm["X_umap"] = umap_embed

In [23]:
def getIMT(prot):
    peaks_E, _ = find_peaks(prot[:,int((win_len*11+win_len/2)-1)], prominence = 40/200, distance = 10)
    peaks_cdc, _ = find_peaks(prot[:,int((win_len*17+win_len/2)-1)], prominence = 40/200, distance = 10)
    
    time_st = 0.1
    
    tmp = np.empty((peaks_E[3],1))
    tmp[:] = np.nan
    tmp_ind = tmp
    div_time = pd.DataFrame(tmp)

    div_inds = pd.DataFrame(tmp)
    ii=0
    
    for n in range(3,len(peaks_E)-3):   #make sure we have divisions at ends 
        tmp_g1_s = peaks_E[n]
        tmp_g1_s_2 = peaks_E[n+1]
        tmp_g1_s_3 = peaks_E[n+2]  
        
        tmp_mit = peaks_cdc[(tmp_g1_s < peaks_cdc) & (peaks_cdc < tmp_g1_s_2)]
        tmp_mit2 = peaks_cdc[(tmp_g1_s_2 < peaks_cdc) & (peaks_cdc < tmp_g1_s_3)]
          
             
        if (np.size(tmp_mit) > 0) & (np.size(tmp_mit2)>0):
            ordered = [tmp_g1_s, tmp_mit[0], tmp_g1_s_2, tmp_mit2[0], tmp_g1_s_3]
            g1_len_tmp = np.array([ordered[2]*time_st]) - np.array([ordered[1]*time_st])
            tmp = [g1_len_tmp]*int(ordered[2]-ordered[0])
            tmp_ind = [np.array([ii])]*int(ordered[2]-ordered[0])
            tmp = np.concatenate(tmp, axis=0)
            tmp_ind = np.concatenate(tmp_ind, axis=0)
            tmp_df = pd.DataFrame(tmp)
            tmp_df_ind = pd.DataFrame(tmp_ind)

            div_time = pd.concat([div_time, tmp_df], ignore_index=True)
            div_inds = pd.concat([div_inds, tmp_df_ind], ignore_index=True)
            ii+=1
        else:
            tmp = np.empty((int(peaks_E[n+1]-peaks_E[n]),1))
            tmp[:] = np.nan
            tmp_df = pd.DataFrame(tmp)
            div_time = pd.concat([div_time, tmp_df], ignore_index=True) 
            div_inds = pd.concat([div_inds, tmp_df], ignore_index=True)
            ii+=1
    
    tmp = np.empty((len(prot)-peaks_E[-3],1))
    tmp[:] = np.nan
    tmp_df = pd.DataFrame(tmp)
    div_time = pd.concat([div_time, tmp_df], ignore_index=True) 
    div_inds = pd.concat([div_inds, tmp_df], ignore_index=True)
    
    return div_time, div_inds

In [24]:
all_divs = []
all_div_inds = []
for num in range(6):
    tmp_div, div_inds = getIMT(prot[num*len_prot:(num+1)*len_prot,:])
    tmp_list = tmp_div[0].values.tolist()
    tmp_ind_list = div_inds[0].values.tolist()
    all_divs = all_divs + tmp_list
    all_div_inds = all_div_inds + tmp_ind_list
    print(num)

0
1
2
3
4
5


In [25]:
for num in range(6):
    tmp_div, div_inds = getIMT(prot_d[num*len_prot:(num+1)*len_prot,:])
    tmp_list = tmp_div[0].values.tolist()
    tmp_ind_list = div_inds[0].values.tolist()
    all_divs = all_divs + tmp_list
    all_div_inds = all_div_inds + tmp_ind_list
    print(num)

0
1
2
3
4
5


In [26]:
G1_cats = []
for val in all_divs:
    if val <= 15:
        tmp = 'Short'
    elif (val > 15) & (val <= 25):
        tmp = 'Int'
    elif (val > 25) & (val <= 50):
        tmp = 'Long'
    elif val > 50:
        tmp = 'Extra Long'
    else:
        tmp = 'nan'
    
    G1_cats.append(tmp)

In [27]:
adata.obs["G1_time"] = pd.Categorical(G1_cats)

In [28]:
adata.write('orig_params_0318_w_cycdi-wELong.h5ad')